# Tutorial 3: Compute persistence images in multiple directories

This note book illustrates how one computes persistence images, which are vectoizations of persistence diagrams.

In [ ]:
using Pkg
Pkg.activate("../.")
Pkg.instantiate() # note: you do not need to run instantiate if you have already run it finished the first or second tutorials

In [ ]:
include("../src/ECM_TDA.jl")

using .ECM_TDA
using PersistenceDiagrams
using DataFrames
using CSV
using DelimitedFiles
using Ripserer

# Compute persistence images for persistence diagrams in a single directory
* In most cases, you won't need to compute this separately. The persistence images are automatically computed when you run any of the following scripts:
    - `compute_persistence.jl`
    - `compute_persistence_script.jl`
    - `compute_Dowker_persistence.jl`
    - `compute_Dowker_persistence_script.jl`


In [ ]:
# load persistence diagrams in a directory, organize them in a dictionary

PDs = Dict()
PD_dir = "PH_outputs/PD/PD1"
for file in readdir(PD_dir)
    filepath = joinpath(PD_dir, file)
    df = CSV.read(filepath, DataFrame; delim = ',', header = [:x, :y])
    arr = Matrix(df)
    PDs[file] = arr
end

Here is what the dictionary looks like

In [ ]:
PDs

The following cell computes the persistence images

In [ ]:
# convert array to Ripserer PD
PH = Dict(k => ECM_TDA.array_to_ripsererPD(v) for (k,v) in PDs if v != nothing)

# compute PI
PersIm = PersistenceImage([PH[k] for k in keys(PH)], sigma=50, size = 20)

PI = Dict()
for i in keys(PH)
    PI[i] = PersIm(PH[i])
end

# Compute persistences images in multiple directories

Sometimes, we'll want to compare the topological features from two different directories. In this case, we want to make sure that the persistence images are computed at a comparable scale. One way to do this is to compute the persistence image in one directory, save the scale parameters involved, and then input the scale parameters when computing the persistence image in the second directory.

In [ ]:
# compute persistence image from one of the directories
PD_dir1 = Dict()
PD_dir = "PI_tutorial/PD1_dir1"
for file in readdir(PD_dir)
    if file != ".DS_Store"
        filepath = joinpath(PD_dir, file)
        df = CSV.read(filepath, DataFrame; delim = ',', header = [:x, :y])
        arr = Matrix(df)
        PD_dir1[file] = arr
    end
end

# convert array to Ripserer PD
PH_dir1 = Dict(k => ECM_TDA.array_to_ripsererPD(v) for (k,v) in PD_dir1 if v != nothing)


# compute PI
PersIm_dir1 = PersistenceImage([PH_dir1[k] for k in keys(PH_dir1)], sigma=50, size = 20)

PI_dir1 = Dict()
for i in keys(PH_dir1)
    PI_dir1[i] = PersIm_dir1(PH_dir1[i])
end

Save the parameters involved

In [ ]:
# get the parameters involved 
PI_xmin = PersIm_dir1.xs[1]
PI_xmax = PersIm_dir1.xs[end]
PI_ymin = PersIm_dir1.ys[1]
PI_ymax = PersIm_dir1.ys[end];

Now compute the persistence images for the persistence diagrams in the second directory.

In [ ]:
# compute persistence image from one of the directories
PD_dir2 = Dict()
PD_dir = "PI_tutorial/PD1_dir2"
for file in readdir(PD_dir)
    if file != ".DS_Store"
        filepath = joinpath(PD_dir, file)
        df = CSV.read(filepath, DataFrame; delim = ',', header = [:x, :y])
        arr = Matrix(df)
        PD_dir2[file] = arr
    end
end

# convert array to Ripserer PD
PH_dir2 = Dict(k => ECM_TDA.array_to_ripsererPD(v) for (k,v) in PD_dir2 if v != nothing)

This time, when we define persistence image, we make sure to pass the parameters

In [ ]:
PersIm_dir2 = PersistenceImage((PI_ymin, PI_ymax),(PI_xmin, PI_xmax), sigma= 50, size = 20)

PI_dir2 = Dict()
for i in keys(PH_dir2)
    PI_dir2[i] = PersIm_dir2(PH_dir2[i])
end

With persistence images we can do further analysis using PCA and/or UMAP, there are examples of this in the Analysis folder